In [1]:
import h5py 
import numpy as np
import pandas as pd
import os
import scipy.signal as signal

In [2]:
def get_dataset_name(data):
    filename = data.split('/')[-1]
    print(f'filename: {filename}')
    temp = filename.split('_')[:-1]
    print(f'temp: {temp}')
    dataset_name = "_".join(temp)
    return dataset_name

In [3]:
file_path = 'dataset/Final Project data/Intra/train/rest_105923_1.h5'

with h5py.File(file_path, 'r')as f:
    dataset01 = get_dataset_name(file_path)
    matrix = f.get(dataset01)[()]
    print(type(matrix))
    print(matrix.shape)

filename: rest_105923_1.h5
temp: ['rest', '105923']
<class 'numpy.ndarray'>
(248, 35624)


In [4]:
print(35624 / 2035)

17.505651105651104


In [5]:
def downsampling_data(data, nr_sample):
    data_downsampled = signal.resample(data, num = nr_sample, axis = 1)

    return  data_downsampled

In [6]:
tasks = ['rest', 'task_motor', 'task_story_math', 'task_working_memory']
# stats = {task: {'sum': np.zeros((248, 1)), 'sq_sum': np.zeros((248, 1)), 'steps': 0} for task in tasks}

def combine_train_arr():
    file_dir = r'dataset\Final Project data\Intra\train'
    nr = '_105923_'
    ini_arrays = {'rest': [], 'task_motor': [], 'task_story_math': [], 
                  'task_working_memory': []}    

    for task in tasks:
        for i in range(1, 9):
            #print(task+nr+str(i)+'.h5')
            file_path = os.path.join(file_dir, task+nr+str(i)+'.h5')
            

            with h5py.File(file_path, 'r')as f:
                file_keys = list(f.keys())
                if not file_keys:
                        print(f"Warning: File {file_path} is empty inside!")
                        continue
                
                dataset01 = file_keys[0]
                
                matrix = f.get(dataset01)[()]
                # print(type(matrix))
                # print(matrix.shape)
                ini_arrays[task].append(matrix)
    return ini_arrays

all_combined_arrs = combine_train_arr()


In [7]:
print(all_combined_arrs)

{'rest': [array([[ 1.97084298e-12,  2.05354267e-12,  2.15993716e-12, ...,
         3.76791523e-12,  3.73353432e-12,  3.84085819e-12],
       [-3.65499341e-12, -3.84306129e-12, -4.03623706e-12, ...,
        -5.83472593e-12, -5.77250097e-12, -5.84865663e-12],
       [ 2.25851130e-12,  2.22853290e-12,  2.31691077e-12, ...,
         3.08775379e-12,  3.10566308e-12,  3.13019098e-12],
       ...,
       [ 2.69611351e-12,  2.78066936e-12,  2.76576960e-12, ...,
         4.65467768e-12,  4.52058052e-12,  4.51164063e-12],
       [-5.07980854e-12, -5.11056778e-12, -5.06082155e-12, ...,
        -6.20575379e-12, -6.26423436e-12, -6.20271586e-12],
       [-6.09371928e-12, -6.22285079e-12, -6.26193498e-12, ...,
        -3.39112787e-12, -3.35855757e-12, -3.36085673e-12]],
      shape=(248, 35624)), array([[ 4.16468769e-12,  4.21207730e-12,  4.14099267e-12, ...,
         5.54966446e-12,  5.70995334e-12,  5.80984349e-12],
       [-6.33995460e-12, -6.45976068e-12, -6.31859365e-12, ...,
        -8.1259174

In [8]:
def z_norm(matrix):
    mean = np.mean(matrix, axis = 1, keepdims = True)
    std = np.std(matrix, axis = 1, keepdims= True)
    matrix_z = (matrix - mean )/std
    
    return matrix_z


In [ ]:
processed_datasets = {}
for task_arr in all_combined_arrs:
    raw_files = all_combined_arrs[task_arr]
    downsampled_files = [downsampling_data(f, 4000) for f in raw_files]

    combined_task_data = np.concatenate(downsampled_files, axis =1)
    task_mean = np.mean(combined_task_data, axis=1, keepdims=True) # Shape: (248, 1)
    task_std = np.std(combined_task_data, axis=1, keepdims=True)  
    task_std = np.where(task_std == 0, 1.0, task_std) # Numerical stability guard
    

    normalized_files = []
    for ds_matrix in downsampled_files:
        norm_matrix = (ds_matrix - task_mean) / task_std
        normalized_files.append(norm_matrix)
        

    processed_datasets[task_arr] = np.array(normalized_files)
    
    print(f"Task: {task_arr:<20} | Final clean shape: {processed_datasets[task_arr].shape}")

Task: rest                 | Final clean shape: (8, 248, 4000)
Task: task_motor           | Final clean shape: (8, 248, 4000)
Task: task_story_math      | Final clean shape: (8, 248, 4000)
Task: task_working_memory  | Final clean shape: (8, 248, 4000)


In [42]:
tasks = ['rest', 'task_motor', 'task_story_math', 'task_working_memory']
# stats = {task: {'sum': np.zeros((248, 1)), 'sq_sum': np.zeros((248, 1)), 'steps': 0} for task in tasks}

def combine_cross_train_arr():
    file_dir = r'dataset\Final Project data\Cross\train'
    nrs = ["_113922_",  "_164636_"]
    ini_arrays = {'rest': [], 'task_motor': [], 'task_story_math': [], 
                  'task_working_memory': []}    

    for task in tasks:
        for nr in nrs:
            for i in range(1, 9):
                #print(task+nr+str(i)+'.h5')
                file_path = os.path.join(file_dir, task+nr+str(i)+'.h5')
                

                with h5py.File(file_path, 'r')as f:
                    file_keys = list(f.keys())
                    if not file_keys:
                            print(f"Warning: File {file_path} is empty inside!")
                            continue
                    
                    dataset01 = file_keys[0]
                    
                    matrix = f.get(dataset01)[()]
                    # print(type(matrix))
                    # print(matrix.shape)
                    ini_arrays[task].append(matrix)
    return ini_arrays

In [43]:
cross_combined_arr = combine_cross_train_arr()

In [44]:
x = cross_combined_arr.values()
print(x)

dict_values([[array([[ 3.29267061e-12,  3.25503752e-12,  3.13702796e-12, ...,
         3.58721386e-12,  3.71823232e-12,  3.88130847e-12],
       [-3.18692924e-12, -3.15210184e-12, -2.96682015e-12, ...,
        -3.33181768e-12, -3.54774738e-12, -3.82172289e-12],
       [ 1.58730494e-12,  1.42845815e-12,  1.32878962e-12, ...,
         2.00427695e-12,  2.00661297e-12,  2.03347669e-12],
       ...,
       [ 2.66890459e-12,  2.63538020e-12,  2.63910508e-12, ...,
         3.08349093e-12,  3.08721581e-12,  3.24105521e-12],
       [-4.54969049e-12, -4.60589249e-12, -4.64462626e-12, ...,
        -4.60246164e-12, -4.54398108e-12, -4.58043629e-12],
       [-3.75057450e-12, -3.75593914e-12, -3.69003222e-12, ...,
        -6.07841034e-12, -6.08109266e-12, -6.06844783e-12]],
      shape=(248, 35624)), array([[ 3.99234899e-12,  3.90546796e-12,  3.76655131e-12, ...,
         4.43232603e-12,  4.37285668e-12,  4.36495848e-12],
       [-3.88766276e-12, -3.66198131e-12, -3.44883779e-12, ...,
        -4.459

In [ ]:
processed_cross_datasets = {}
for task_arr in cross_combined_arr:
    raw_files = cross_combined_arr[task_arr]

    downsampled_files = [downsampling_data(f, 4000) for f in raw_files]
    
    combined_cross_task_data = np.concatenate(downsampled_files, axis =1)
    task_mean = np.mean(combined_cross_task_data, axis=1, keepdims=True) # Shape: (248, 1)
    task_std = np.std(combined_cross_task_data, axis=1, keepdims=True)   
    task_std = np.where(task_std == 0, 1.0, task_std) # Numerical stability guard

    normalized_files = []
    for ds_matrix in downsampled_files:
        norm_matrix = (ds_matrix - task_mean) / task_std
        normalized_files.append(norm_matrix)
        

    processed_datasets[task_arr] = np.array(normalized_files)
    
    print(f"Task: {task_arr:<20} | Final clean shape: {processed_datasets[task_arr].shape}")

Task: rest                 | Final clean shape: (16, 248, 4000)
Task: task_motor           | Final clean shape: (16, 248, 4000)
Task: task_story_math      | Final clean shape: (16, 248, 4000)
Task: task_working_memory  | Final clean shape: (16, 248, 4000)
